# 🤖 AgriSense AI — Model Training & Evaluation
This notebook trains the **Random Forest Classifier** for Crop Recommendation and Fertilizer Prediction,
evaluates performance metrics (Accuracy, F1-Score, Classification Report), and exports model artifacts.

In [1]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Modules imported successfully.")

## 1. Train Crop Recommendation Model (Random Forest)

In [2]:
crop_df = pd.read_csv("../ml_models/datasets/Crop_recommendation.csv")
features = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
X = crop_df[features]
y = crop_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf_crop = RandomForestClassifier(n_estimators=100, random_state=42)
rf_crop.fit(X_train_scaled, y_train)

y_pred = rf_crop.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"🎯 Crop Model Test Accuracy: {acc * 100:.2f}%")
print("
Classification Report:
", classification_report(y_test, y_pred))

## 2. Feature Importance Analysis
Inspect the most influential environmental and soil parameters in crop suitability.

In [3]:
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf_crop.feature_importances_
}).sort_values(by="Importance", ascending=False)
print(importance_df.to_string(index=False))

## 3. Train Fertilizer Model & Evaluation

In [4]:
fert_df = pd.read_csv("../ml_models/datasets/Fertilizer Prediction.csv")
label_encoders = {}
for col in ["Soil Type", "Crop Type"]:
    le = LabelEncoder()
    fert_df[col] = le.fit_transform(fert_df[col])
    label_encoders[col] = le

X_fert = fert_df.drop(columns=["Fertilizer Name"])
y_fert = fert_df["Fertilizer Name"]

X_f_train, X_f_test, y_f_train, y_f_test = train_test_split(
    X_fert, y_fert, test_size=0.2, random_state=42, stratify=y_fert
)

fert_scaler = StandardScaler()
X_f_train_scaled = fert_scaler.fit_transform(X_f_train)
X_f_test_scaled = fert_scaler.transform(X_f_test)

rf_fert = RandomForestClassifier(n_estimators=100, random_state=42)
rf_fert.fit(X_f_train_scaled, y_f_train)
fert_pred = rf_fert.predict(X_f_test_scaled)
fert_acc = accuracy_score(y_f_test, fert_pred)
print(f"🎯 Fertilizer Model Holdout Accuracy: {fert_acc * 100:.2f}% (on {len(y_f_test)} test samples)")